# ISL Sign Language Recognition: Pose + MobileNetV2 + BiLSTM

## Overview
This notebook implements a **multimodal** Indian Sign Language (ISL) recognition pipeline using the **INCLUDE** dataset (263 classes).

### Architecture
- **Pose keypoints**: MediaPipe extracts 25 pose + 21+21 hand landmarks = **134 keypoints** per frame
- **CNN features**: Pretrained MobileNetV2 extracts **1280-dim** visual features per frame
- **Fusion**: Concatenate keypoints + CNN features = **1414-dim** per frame
- **Sequence model**: BiLSTM (2 layers, bidirectional) processes 30-frame sequences

### Pipeline
1. Extract zip files containing INCLUDE videos
2. Run MediaPipe + MobileNetV2 feature extraction, save `.npz` files to Drive
3. Train BiLSTM classifier with AdamW + cosine LR + warmup
4. Evaluate and visualize results

**Dataset**: INCLUDE (263 ISL signs)
**Sequence length**: 30 frames (uniform sampling)
**Batch size**: 32 | **Epochs**: 100 | **Patience**: 20


In [ ]:
# Cell 1: Clone repo and install dependencies
import subprocess, sys

def run(cmd):
    result = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    if result.stdout: print(result.stdout[-2000:])
    if result.stderr: print(result.stderr[-1000:])
    return result.returncode

# Clone the INCLUDE repo (contains label maps and train/test splits)
run("git clone --depth 1 https://github.com/your-org/INCLUDE.git /content/INCLUDE 2>/dev/null || echo 'Repo already cloned or not available'")

# Install required packages
packages = [
    "mediapipe>=0.10.0",
    "torch torchvision",
    "timm",
    "scikit-learn",
    "matplotlib seaborn",
    "tqdm",
    "opencv-python-headless",
]
for pkg in packages:
    run(f"pip install -q {pkg}")

print("All dependencies installed.")


In [ ]:
# Cell 2: Mount Google Drive and configure all paths
from google.colab import drive
drive.mount('/content/drive')

import os

# ── Configurable paths ──────────────────────────────────────────────────────
# Zip files on Drive containing INCLUDE videos (update these paths)
DRIVE_ROOT        = '/content/drive/MyDrive'
ZIP_TRAIN         = os.path.join(DRIVE_ROOT, 'INCLUDE_train.zip')
ZIP_VAL           = os.path.join(DRIVE_ROOT, 'INCLUDE_val.zip')
ZIP_TEST          = os.path.join(DRIVE_ROOT, 'INCLUDE_test.zip')

# Where to extract videos locally
VIDEO_DIR         = '/content/videos'

# Where to save extracted .npz feature files on Drive
FEATURES_TRAIN_DIR = os.path.join(DRIVE_ROOT, 'include_features/train')
FEATURES_VAL_DIR   = os.path.join(DRIVE_ROOT, 'include_features/val')
FEATURES_TEST_DIR  = os.path.join(DRIVE_ROOT, 'include_features/test')

# Checkpoint path on Drive
CHECKPOINT_PATH   = os.path.join(DRIVE_ROOT, 'bilstm_checkpoint.pth')

# Label map path (from cloned repo or Drive)
LABEL_MAP_PATH    = '/content/INCLUDE/label_maps/label_map_include.json'

# Training hyperparameters
MAX_SEQ_LEN  = 30
BATCH_SIZE   = 32
NUM_EPOCHS   = 100
PATIENCE     = 20
LR           = 1e-3
NUM_CLASSES  = 263

# Create local dirs
for d in [VIDEO_DIR, FEATURES_TRAIN_DIR, FEATURES_VAL_DIR, FEATURES_TEST_DIR]:
    os.makedirs(d, exist_ok=True)

print("Drive mounted. Paths configured:")
print(f"  ZIP_TRAIN         : {ZIP_TRAIN}")
print(f"  ZIP_VAL           : {ZIP_VAL}")
print(f"  ZIP_TEST          : {ZIP_TEST}")
print(f"  FEATURES_TRAIN_DIR: {FEATURES_TRAIN_DIR}")
print(f"  CHECKPOINT_PATH   : {CHECKPOINT_PATH}")


In [ ]:
# Cell 3: Extract zip files to /content/videos/
import zipfile, os
from tqdm.auto import tqdm

def extract_zip(zip_path, extract_to):
    if not os.path.exists(zip_path):
        print(f"WARNING: {zip_path} not found, skipping.")
        return
    print(f"Extracting {zip_path} -> {extract_to}")
    with zipfile.ZipFile(zip_path, 'r') as zf:
        members = zf.namelist()
        for member in tqdm(members, desc=os.path.basename(zip_path)):
            zf.extract(member, extract_to)
    print(f"  Done. {len(members)} files extracted.")

extract_zip(ZIP_TRAIN, VIDEO_DIR)
extract_zip(ZIP_VAL,   VIDEO_DIR)
extract_zip(ZIP_TEST,  VIDEO_DIR)

# Count extracted videos
import glob
video_files = glob.glob(os.path.join(VIDEO_DIR, '**', '*.MOV'), recursive=True)
video_files += glob.glob(os.path.join(VIDEO_DIR, '**', '*.mp4'), recursive=True)
video_files += glob.glob(os.path.join(VIDEO_DIR, '**', '*.avi'), recursive=True)
print(f"Total video files found: {len(video_files)}")


In [ ]:
# Cell 4: Define MediaPipe keypoint extractor
# Extracts pose + hand keypoints per frame from a video
# Returns numpy array of shape (n_frames, 134)
# Layout: 25 pose (x,y) + 21 hand1 (x,y) + 21 hand2 (x,y) = 134 values

import cv2
import numpy as np
import mediapipe as mp

mp_pose  = mp.solutions.pose
mp_hands = mp.solutions.hands

def uniform_sample_indices(n_frames, target=30):
    """Return `target` evenly-spaced frame indices from a video with n_frames."""
    if n_frames <= target:
        return list(range(n_frames))
    step = n_frames / target
    return [int(i * step) for i in range(target)]

def extract_keypoints(video_path, target_frames=30):
    """
    Extract pose + hand keypoints from a video using MediaPipe.

    Parameters
    ----------
    video_path : str
        Path to the video file.
    target_frames : int
        Number of frames to uniformly sample (default 30).

    Returns
    -------
    np.ndarray of shape (target_frames, 134)
        Each row: [pose_x0, pose_y0, ..., pose_x24, pose_y24,
                   hand1_x0, hand1_y0, ..., hand1_x20, hand1_y20,
                   hand2_x0, hand2_y0, ..., hand2_x20, hand2_y20]
        Missing landmarks are filled with 0.0.
    """
    cap = cv2.VideoCapture(video_path)
    if not cap.isOpened():
        raise IOError(f"Cannot open video: {video_path}")

    # Read all frames first to allow uniform sampling
    frames = []
    while True:
        ret, frame = cap.read()
        if not ret:
            break
        frames.append(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))
    cap.release()

    if len(frames) == 0:
        return np.zeros((target_frames, 134), dtype=np.float32)

    indices = uniform_sample_indices(len(frames), target_frames)

    pose_model  = mp_pose.Pose(
        static_image_mode=False,
        min_detection_confidence=0.5,
        min_tracking_confidence=0.5
    )
    hands_model = mp_hands.Hands(
        static_image_mode=False,
        max_num_hands=2,
        min_detection_confidence=0.5,
        min_tracking_confidence=0.5
    )

    keypoints = []
    for idx in indices:
        frame = frames[idx]

        # Pose: 25 landmarks (upper body) -> 50 values
        pose_res = pose_model.process(frame)
        if pose_res.pose_landmarks:
            lms = pose_res.pose_landmarks.landmark[:25]
            pose_kp = np.array([[lm.x, lm.y] for lm in lms], dtype=np.float32).flatten()
        else:
            pose_kp = np.zeros(50, dtype=np.float32)

        # Hands: 2 x 21 landmarks -> 84 values
        hand_res = hands_model.process(frame)
        h1_kp = np.zeros(42, dtype=np.float32)
        h2_kp = np.zeros(42, dtype=np.float32)
        if hand_res.multi_hand_landmarks:
            h1 = hand_res.multi_hand_landmarks[0]
            h1_kp = np.array([[lm.x, lm.y] for lm in h1.landmark], dtype=np.float32).flatten()
            if len(hand_res.multi_hand_landmarks) > 1:
                h2 = hand_res.multi_hand_landmarks[1]
                h2_kp = np.array([[lm.x, lm.y] for lm in h2.landmark], dtype=np.float32).flatten()

        # Concatenate: 50 + 42 + 42 = 134
        frame_kp = np.concatenate([pose_kp, h1_kp, h2_kp])
        keypoints.append(frame_kp)

    pose_model.close()
    hands_model.close()

    # Pad if fewer frames than target
    while len(keypoints) < target_frames:
        keypoints.append(np.zeros(134, dtype=np.float32))

    result = np.stack(keypoints[:target_frames], axis=0)  # (target_frames, 134)
    assert result.shape == (target_frames, 134), f"Unexpected shape: {result.shape}"
    return result

print("extract_keypoints() defined.")
print("Output shape per video: (30, 134)")
print("  - 25 pose landmarks x 2 (x,y) = 50")
print("  - 21 hand1 landmarks x 2 (x,y) = 42")
print("  - 21 hand2 landmarks x 2 (x,y) = 42")
print("  Total: 134 keypoints per frame")


In [ ]:
# Cell 5: Define MobileNetV2 feature extractor
# Loads pretrained MobileNetV2, removes classifier head, extracts 1280-dim features per frame

import torch
import torchvision.models as models
import torchvision.transforms as T
from PIL import Image
import numpy as np
import cv2

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {DEVICE}")

# Load MobileNetV2 pretrained on ImageNet, strip classifier head
_mobilenet = models.mobilenet_v2(weights=models.MobileNet_V2_Weights.IMAGENET1K_V1)
_mobilenet.classifier = torch.nn.Identity()  # Remove final classifier -> outputs 1280-dim
_mobilenet = _mobilenet.to(DEVICE)
_mobilenet.eval()

# ImageNet normalization
_cnn_transform = T.Compose([
    T.Resize((224, 224)),
    T.ToTensor(),
    T.Normalize(mean=[0.485, 0.456, 0.406],
                std=[0.229, 0.224, 0.225]),
])

def extract_cnn_features(video_path, target_frames=30, batch_size=8):
    """
    Extract MobileNetV2 features from uniformly sampled frames.

    Parameters
    ----------
    video_path : str
        Path to the video file.
    target_frames : int
        Number of frames to uniformly sample (default 30).
    batch_size : int
        Frames processed per forward pass (default 8).

    Returns
    -------
    np.ndarray of shape (target_frames, 1280)
        MobileNetV2 penultimate-layer features per frame.
    """
    cap = cv2.VideoCapture(video_path)
    if not cap.isOpened():
        raise IOError(f"Cannot open video: {video_path}")

    frames = []
    while True:
        ret, frame = cap.read()
        if not ret:
            break
        frames.append(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))
    cap.release()

    if len(frames) == 0:
        return np.zeros((target_frames, 1280), dtype=np.float32)

    indices = uniform_sample_indices(len(frames), target_frames)
    sampled = [frames[i] for i in indices]

    # Pad if needed
    while len(sampled) < target_frames:
        sampled.append(np.zeros_like(sampled[0]))

    # Convert to tensors
    tensors = [_cnn_transform(Image.fromarray(f)) for f in sampled[:target_frames]]

    features = []
    with torch.no_grad():
        for i in range(0, len(tensors), batch_size):
            batch = torch.stack(tensors[i:i+batch_size]).to(DEVICE)
            feat = _mobilenet(batch)  # (B, 1280)
            features.append(feat.cpu().numpy())

    result = np.concatenate(features, axis=0)  # (target_frames, 1280)
    assert result.shape == (target_frames, 1280), f"Unexpected shape: {result.shape}"
    return result

print("extract_cnn_features() defined.")
print("Output shape per video: (30, 1280)")
print(f"MobileNetV2 loaded on {DEVICE}")


In [ ]:
# Cell 6: Feature extraction pipeline
# For each video: extract keypoints + CNN features, save as .npz to Drive
# Includes resume logic: skips already-processed videos

import os, glob, json
import numpy as np
from tqdm.auto import tqdm

def get_label_from_path(video_path):
    """Extract label from directory name (e.g. 'Adjectives/6. Ugly/MVI_9579.MOV' -> 'ugly')."""
    parts = video_path.replace('\\', '/').split('/')
    # Label is the parent directory name, strip leading numbers and dots
    label_dir = parts[-2]
    label = ''.join([c for c in label_dir if c.isalpha()]).lower()
    return label

def get_uid(video_path):
    """Generate unique ID: label_filename (no extension)."""
    label = get_label_from_path(video_path)
    fname = os.path.splitext(os.path.basename(video_path))[0]
    return f"{label}_{fname}"

def process_split(video_paths, save_dir, label_map, split_name='train'):
    """
    Extract and save features for all videos in a split.
    Saves one .npz per video with keys: 'keypoints' (30,134), 'cnn' (30,1280), 'label' (int).
    Skips videos already processed (resume logic).
    """
    os.makedirs(save_dir, exist_ok=True)
    skipped = 0
    processed = 0
    errors = 0

    for vpath in tqdm(video_paths, desc=f"Extracting {split_name}"):
        label = get_label_from_path(vpath)
        if label not in label_map:
            errors += 1
            continue

        uid = get_uid(vpath)
        save_path = os.path.join(save_dir, f"{uid}.npz")

        # Resume: skip if already processed
        if os.path.exists(save_path):
            skipped += 1
            continue

        if not os.path.exists(vpath):
            errors += 1
            continue

        try:
            kp   = extract_keypoints(vpath, target_frames=MAX_SEQ_LEN)   # (30, 134)
            cnn  = extract_cnn_features(vpath, target_frames=MAX_SEQ_LEN) # (30, 1280)
            lbl  = label_map[label]
            np.savez_compressed(save_path, keypoints=kp, cnn=cnn, label=np.array(lbl))
            processed += 1
        except Exception as e:
            print(f"  ERROR processing {vpath}: {e}")
            errors += 1

    print(f"[{split_name}] Processed: {processed} | Skipped (resume): {skipped} | Errors: {errors}")
    return processed, skipped, errors

# Load label map
with open(LABEL_MAP_PATH, 'r') as f:
    label_map = json.load(f)
print(f"Label map loaded: {len(label_map)} classes")

# Collect video paths from extracted directories
def collect_videos(root_dir):
    paths = []
    for ext in ['*.MOV', '*.mp4', '*.avi', '*.mov']:
        paths += glob.glob(os.path.join(root_dir, '**', ext), recursive=True)
    return sorted(paths)

# Load official train/val/test splits from text files
def load_split_paths(split_file, video_root):
    """Load video paths from INCLUDE split text file."""
    if not os.path.exists(split_file):
        print(f"Split file not found: {split_file}, falling back to directory scan")
        return collect_videos(video_root)
    with open(split_file) as f:
        lines = [l.strip() for l in f if l.strip()]
    paths = [os.path.join(video_root, l) for l in lines]
    return paths

SPLIT_DIR = '/content/INCLUDE/train_test_paths'
train_paths = load_split_paths(os.path.join(SPLIT_DIR, 'include_train.txt'), VIDEO_DIR)
val_paths   = load_split_paths(os.path.join(SPLIT_DIR, 'include_val.txt'),   VIDEO_DIR)
test_paths  = load_split_paths(os.path.join(SPLIT_DIR, 'include_test.txt'),  VIDEO_DIR)

print(f"Train videos: {len(train_paths)}")
print(f"Val   videos: {len(val_paths)}")
print(f"Test  videos: {len(test_paths)}")

# Run extraction
process_split(train_paths, FEATURES_TRAIN_DIR, label_map, 'train')
process_split(val_paths,   FEATURES_VAL_DIR,   label_map, 'val')
process_split(test_paths,  FEATURES_TEST_DIR,  label_map, 'test')

print("Feature extraction complete!")


In [ ]:
# Cell 7: Verify extracted features
import os, glob
import numpy as np

def verify_split(features_dir, split_name):
    files = sorted(glob.glob(os.path.join(features_dir, '*.npz')))
    print(f"\n[{split_name}] {len(files)} .npz files in {features_dir}")
    if not files:
        print("  WARNING: No files found!")
        return

    # Check a few random files
    import random
    samples = random.sample(files, min(5, len(files)))
    for fp in samples:
        data = np.load(fp)
        kp_shape  = data['keypoints'].shape
        cnn_shape = data['cnn'].shape
        label     = int(data['label'])
        ok_kp  = kp_shape  == (MAX_SEQ_LEN, 134)
        ok_cnn = cnn_shape == (MAX_SEQ_LEN, 1280)
        status = 'OK' if (ok_kp and ok_cnn) else 'SHAPE ERROR'
        print(f"  {status} | {os.path.basename(fp)} | kp={kp_shape} cnn={cnn_shape} label={label}")

    # Count unique labels
    labels = set()
    for fp in files:
        d = np.load(fp)
        labels.add(int(d['label']))
    print(f"  Unique labels: {len(labels)}")

verify_split(FEATURES_TRAIN_DIR, 'train')
verify_split(FEATURES_VAL_DIR,   'val')
verify_split(FEATURES_TEST_DIR,  'test')


In [ ]:
# Cell 8: Dataset class
# Loads .npz files, pads/truncates to max_seq_len=30 frames
# Returns (fused_features, label) where fused = concat(keypoints, cnn) = 1414 dims

import os, glob
import numpy as np
import torch
from torch.utils.data import Dataset, DataLoader

class ISLFusedDataset(Dataset):
    """
    Dataset for ISL sign recognition using fused keypoint + CNN features.

    Each .npz file contains:
        keypoints : (30, 134)  - MediaPipe pose+hand keypoints
        cnn       : (30, 1280) - MobileNetV2 features
        label     : int        - class index

    Returns:
        features : torch.FloatTensor of shape (max_seq_len, 1414)
        label    : torch.LongTensor scalar
    """

    KEYPOINT_DIM = 134
    CNN_DIM      = 1280
    FUSED_DIM    = 134 + 1280  # 1414

    def __init__(self, features_dir, max_seq_len=30, augment=False):
        self.max_seq_len = max_seq_len
        self.augment     = augment
        self.files = sorted(glob.glob(os.path.join(features_dir, '*.npz')))
        if not self.files:
            raise RuntimeError(f"No .npz files found in {features_dir}")
        print(f"ISLFusedDataset: {len(self.files)} samples from {features_dir}")

    def __len__(self):
        return len(self.files)

    def _pad_or_truncate(self, arr, target_len, feat_dim):
        """Ensure array is exactly (target_len, feat_dim)."""
        n = arr.shape[0]
        if n >= target_len:
            return arr[:target_len]
        # Pad with zeros
        pad = np.zeros((target_len - n, feat_dim), dtype=np.float32)
        return np.concatenate([arr, pad], axis=0)

    def __getitem__(self, idx):
        data = np.load(self.files[idx])
        kp  = data['keypoints'].astype(np.float32)  # (T, 134)
        cnn = data['cnn'].astype(np.float32)         # (T, 1280)
        lbl = int(data['label'])

        # Pad/truncate to max_seq_len
        kp  = self._pad_or_truncate(kp,  self.max_seq_len, self.KEYPOINT_DIM)
        cnn = self._pad_or_truncate(cnn, self.max_seq_len, self.CNN_DIM)

        # Fuse: concatenate along feature dimension -> (max_seq_len, 1414)
        fused = np.concatenate([kp, cnn], axis=-1)

        # Normalize per-sample (zero mean, unit std)
        mean = fused.mean()
        std  = fused.std() + 1e-8
        fused = (fused - mean) / std

        # Optional: simple augmentation (Gaussian noise on keypoints)
        if self.augment:
            noise = np.random.randn(*fused.shape).astype(np.float32) * 0.01
            fused = fused + noise

        return torch.FloatTensor(fused), torch.tensor(lbl, dtype=torch.long)


# Build datasets and dataloaders
train_dataset = ISLFusedDataset(FEATURES_TRAIN_DIR, max_seq_len=MAX_SEQ_LEN, augment=True)
val_dataset   = ISLFusedDataset(FEATURES_VAL_DIR,   max_seq_len=MAX_SEQ_LEN, augment=False)
test_dataset  = ISLFusedDataset(FEATURES_TEST_DIR,  max_seq_len=MAX_SEQ_LEN, augment=False)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True,
                          num_workers=2, pin_memory=True, drop_last=True)
val_loader   = DataLoader(val_dataset,   batch_size=BATCH_SIZE, shuffle=False,
                          num_workers=2, pin_memory=True)
test_loader  = DataLoader(test_dataset,  batch_size=BATCH_SIZE, shuffle=False,
                          num_workers=2, pin_memory=True)

# Verify a batch
sample_feat, sample_lbl = next(iter(train_loader))
print(f"Sample batch - features: {sample_feat.shape}, labels: {sample_lbl.shape}")
print(f"  Feature dim: {sample_feat.shape[-1]} (expected 1414 = 134 + 1280)")
assert sample_feat.shape[-1] == 1414, "Fused feature dim mismatch!"
print("Dataset and DataLoaders ready.")


In [ ]:
# Cell 9: BiLSTM model definition
# Architecture: Linear(1414->512) -> BiLSTM(512,256,2L) -> Dropout(0.3) -> Linear(512->n_classes)

import torch
import torch.nn as nn
import torch.nn.functional as F

class MultimodalBiLSTM(nn.Module):
    """
    Multimodal BiLSTM for ISL sign recognition.

    Architecture:
        1. Input projection: Linear(1414, 512) + LayerNorm + ReLU
        2. BiLSTM: 2 layers, hidden=256, bidirectional -> output dim=512
        3. Temporal pooling: max-pool over time
        4. Dropout(0.3)
        5. Classifier: Linear(512, n_classes)

    Input:  (batch, seq_len, 1414)
    Output: (batch, n_classes) logits
    """

    def __init__(self, input_dim=1414, proj_dim=512, lstm_hidden=256,
                 lstm_layers=2, n_classes=263, dropout=0.3):
        super().__init__()

        # Input projection
        self.input_proj = nn.Sequential(
            nn.Linear(input_dim, proj_dim),
            nn.LayerNorm(proj_dim),
            nn.ReLU(inplace=True),
        )

        # Bidirectional LSTM
        self.bilstm = nn.LSTM(
            input_size=proj_dim,
            hidden_size=lstm_hidden,
            num_layers=lstm_layers,
            batch_first=True,
            bidirectional=True,
            dropout=dropout if lstm_layers > 1 else 0.0,
        )

        # Output dim of BiLSTM = lstm_hidden * 2 (bidirectional)
        lstm_out_dim = lstm_hidden * 2  # 512

        self.dropout    = nn.Dropout(dropout)
        self.classifier = nn.Linear(lstm_out_dim, n_classes)

        self._init_weights()

    def _init_weights(self):
        for name, param in self.bilstm.named_parameters():
            if 'weight_ih' in name:
                nn.init.xavier_uniform_(param.data)
            elif 'weight_hh' in name:
                nn.init.orthogonal_(param.data)
            elif 'bias' in name:
                param.data.fill_(0)
                # Set forget gate bias to 1
                n = param.size(0)
                param.data[n//4:n//2].fill_(1)
        nn.init.xavier_uniform_(self.classifier.weight)
        nn.init.zeros_(self.classifier.bias)

    def forward(self, x):
        """
        x: (batch, seq_len, 1414)
        returns: (batch, n_classes) logits
        """
        # Project input
        x = self.input_proj(x)          # (B, T, 512)

        # BiLSTM
        x, _ = self.bilstm(x)           # (B, T, 512)

        # Max pooling over time dimension
        x = torch.max(x, dim=1).values  # (B, 512)

        # Dropout + classify
        x = self.dropout(x)
        x = self.classifier(x)          # (B, n_classes)
        return x


# Instantiate model
model = MultimodalBiLSTM(
    input_dim=ISLFusedDataset.FUSED_DIM,  # 1414
    proj_dim=512,
    lstm_hidden=256,
    lstm_layers=2,
    n_classes=NUM_CLASSES,
    dropout=0.3,
).to(DEVICE)

# Print model summary
total_params = sum(p.numel() for p in model.parameters())
trainable    = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(model)
print(f"\nTotal parameters    : {total_params:,}")
print(f"Trainable parameters: {trainable:,}")

# Quick forward pass test
with torch.no_grad():
    dummy = torch.randn(2, MAX_SEQ_LEN, ISLFusedDataset.FUSED_DIM).to(DEVICE)
    out   = model(dummy)
    print(f"\nForward pass test: input {dummy.shape} -> output {out.shape}")
    assert out.shape == (2, NUM_CLASSES), f"Output shape mismatch: {out.shape}"
print("Model OK.")


In [ ]:
# Cell 10: Training setup
# AdamW optimizer, cosine LR with warmup, early stopping, resume from Drive checkpoint

import torch
import torch.nn as nn
import math

# ── Optimizer ──────────────────────────────────────────────────────────────
optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=LR,
    weight_decay=1e-4,
    betas=(0.9, 0.999),
)

# ── Loss ───────────────────────────────────────────────────────────────────
criterion = nn.CrossEntropyLoss(label_smoothing=0.1)

# ── LR Scheduler: cosine with linear warmup ────────────────────────────────
WARMUP_EPOCHS = 5

def get_lr(epoch, warmup_epochs=WARMUP_EPOCHS, total_epochs=NUM_EPOCHS, base_lr=LR):
    """Linear warmup then cosine annealing."""
    if epoch < warmup_epochs:
        return base_lr * (epoch + 1) / warmup_epochs
    progress = (epoch - warmup_epochs) / max(1, total_epochs - warmup_epochs)
    return base_lr * 0.5 * (1.0 + math.cos(math.pi * progress))

scheduler = torch.optim.lr_scheduler.LambdaLR(
    optimizer,
    lr_lambda=lambda epoch: get_lr(epoch) / LR
)

# ── Early stopping state ───────────────────────────────────────────────────
class EarlyStopping:
    def __init__(self, patience=20, min_delta=1e-4):
        self.patience   = patience
        self.min_delta  = min_delta
        self.best_loss  = float('inf')
        self.counter    = 0
        self.best_epoch = 0

    def step(self, val_loss, epoch):
        if val_loss < self.best_loss - self.min_delta:
            self.best_loss  = val_loss
            self.counter    = 0
            self.best_epoch = epoch
            return False  # don't stop
        self.counter += 1
        return self.counter >= self.patience  # stop if True

early_stopper = EarlyStopping(patience=PATIENCE)

# ── Resume from checkpoint ─────────────────────────────────────────────────
start_epoch   = 0
history       = {'train_loss': [], 'val_loss': [], 'train_acc': [], 'val_acc': [], 'lr': []}

if os.path.exists(CHECKPOINT_PATH):
    print(f"Resuming from checkpoint: {CHECKPOINT_PATH}")
    ckpt = torch.load(CHECKPOINT_PATH, map_location=DEVICE)
    model.load_state_dict(ckpt['model_state_dict'])
    optimizer.load_state_dict(ckpt['optimizer_state_dict'])
    scheduler.load_state_dict(ckpt['scheduler_state_dict'])
    start_epoch = ckpt['epoch'] + 1
    history     = ckpt.get('history', history)
    early_stopper.best_loss  = ckpt.get('best_val_loss', float('inf'))
    early_stopper.best_epoch = ckpt.get('best_epoch', 0)
    print(f"  Resumed at epoch {start_epoch}, best val_loss={early_stopper.best_loss:.4f}")
else:
    print("No checkpoint found. Starting fresh training.")

print(f"\nTraining setup:")
print(f"  Optimizer  : AdamW (lr={LR}, weight_decay=1e-4)")
print(f"  Scheduler  : Cosine with {WARMUP_EPOCHS}-epoch warmup")
print(f"  Loss       : CrossEntropyLoss (label_smoothing=0.1)")
print(f"  Early stop : patience={PATIENCE}")
print(f"  Start epoch: {start_epoch}/{NUM_EPOCHS}")


In [ ]:
# Cell 11: Training loop with Drive checkpoint save every epoch
import time

def train_one_epoch(model, loader, optimizer, criterion, device):
    model.train()
    total_loss, correct, total = 0.0, 0, 0
    for features, labels in loader:
        features, labels = features.to(device), labels.to(device)
        optimizer.zero_grad()
        logits = model(features)
        loss   = criterion(logits, labels)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        total_loss += loss.item() * labels.size(0)
        preds       = logits.argmax(dim=1)
        correct    += (preds == labels).sum().item()
        total      += labels.size(0)
    return total_loss / total, correct / total

@torch.no_grad()
def evaluate(model, loader, criterion, device):
    model.eval()
    total_loss, correct, total = 0.0, 0, 0
    for features, labels in loader:
        features, labels = features.to(device), labels.to(device)
        logits = model(features)
        loss   = criterion(logits, labels)
        total_loss += loss.item() * labels.size(0)
        preds       = logits.argmax(dim=1)
        correct    += (preds == labels).sum().item()
        total      += labels.size(0)
    return total_loss / total, correct / total

print(f"Starting training from epoch {start_epoch} to {NUM_EPOCHS-1}")
print("-" * 70)

for epoch in range(start_epoch, NUM_EPOCHS):
    t0 = time.time()

    train_loss, train_acc = train_one_epoch(model, train_loader, optimizer, criterion, DEVICE)
    val_loss,   val_acc   = evaluate(model, val_loader, criterion, DEVICE)
    scheduler.step()

    current_lr = optimizer.param_groups[0]['lr']
    elapsed    = time.time() - t0

    history['train_loss'].append(train_loss)
    history['val_loss'].append(val_loss)
    history['train_acc'].append(train_acc)
    history['val_acc'].append(val_acc)
    history['lr'].append(current_lr)

    print(f"Epoch {epoch+1:3d}/{NUM_EPOCHS} | "
          f"train_loss={train_loss:.4f} acc={train_acc:.3f} | "
          f"val_loss={val_loss:.4f} acc={val_acc:.3f} | "
          f"lr={current_lr:.2e} | {elapsed:.1f}s")

    # Save checkpoint to Drive every epoch
    torch.save({
        'epoch'               : epoch,
        'model_state_dict'    : model.state_dict(),
        'optimizer_state_dict': optimizer.state_dict(),
        'scheduler_state_dict': scheduler.state_dict(),
        'history'             : history,
        'best_val_loss'       : early_stopper.best_loss,
        'best_epoch'          : early_stopper.best_epoch,
        'val_loss'            : val_loss,
        'val_acc'             : val_acc,
    }, CHECKPOINT_PATH)

    # Early stopping
    if early_stopper.step(val_loss, epoch):
        print(f"\nEarly stopping at epoch {epoch+1}. "
              f"Best val_loss={early_stopper.best_loss:.4f} at epoch {early_stopper.best_epoch+1}.")
        break

print("\nTraining complete!")
print(f"Best val_loss: {early_stopper.best_loss:.4f} at epoch {early_stopper.best_epoch+1}")


In [ ]:
# Cell 12: Training curves plot
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams['figure.dpi'] = 120

epochs_range = range(1, len(history['train_loss']) + 1)

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle('Training Curves - Multimodal BiLSTM (ISL)', fontsize=14, fontweight='bold')

# Loss
axes[0].plot(epochs_range, history['train_loss'], label='Train Loss', color='steelblue')
axes[0].plot(epochs_range, history['val_loss'],   label='Val Loss',   color='coral')
axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('Loss')
axes[0].set_title('Loss'); axes[0].legend(); axes[0].grid(alpha=0.3)

# Accuracy
axes[1].plot(epochs_range, [a*100 for a in history['train_acc']], label='Train Acc', color='steelblue')
axes[1].plot(epochs_range, [a*100 for a in history['val_acc']],   label='Val Acc',   color='coral')
axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('Accuracy (%)')
axes[1].set_title('Accuracy'); axes[1].legend(); axes[1].grid(alpha=0.3)

# Learning rate
axes[2].plot(epochs_range, history['lr'], color='green')
axes[2].set_xlabel('Epoch'); axes[2].set_ylabel('Learning Rate')
axes[2].set_title('Learning Rate Schedule'); axes[2].grid(alpha=0.3)
axes[2].set_yscale('log')

plt.tight_layout()
plt.savefig('/content/training_curves.png', bbox_inches='tight')
plt.show()
print("Training curves saved to /content/training_curves.png")


In [ ]:
# Cell 13: Full evaluation - accuracy, F1, confusion matrix
import numpy as np
import torch
from sklearn.metrics import (accuracy_score, f1_score,
                              classification_report, confusion_matrix)
import matplotlib.pyplot as plt
import seaborn as sns
import json

# Load best checkpoint for evaluation
if os.path.exists(CHECKPOINT_PATH):
    ckpt = torch.load(CHECKPOINT_PATH, map_location=DEVICE)
    model.load_state_dict(ckpt['model_state_dict'])
    print(f"Loaded checkpoint from epoch {ckpt['epoch']+1}")

model.eval()

def get_predictions(loader, device):
    all_preds, all_labels = [], []
    with torch.no_grad():
        for features, labels in loader:
            features = features.to(device)
            logits   = model(features)
            preds    = logits.argmax(dim=1).cpu().numpy()
            all_preds.extend(preds)
            all_labels.extend(labels.numpy())
    return np.array(all_preds), np.array(all_labels)

print("Evaluating on test set...")
test_preds, test_labels = get_predictions(test_loader, DEVICE)

# Load label map for class names
with open(LABEL_MAP_PATH) as f:
    label_map = json.load(f)
idx_to_label = {v: k for k, v in label_map.items()}
class_names  = [idx_to_label.get(i, str(i)) for i in range(NUM_CLASSES)]

# Metrics
test_acc = accuracy_score(test_labels, test_preds)
test_f1_macro = f1_score(test_labels, test_preds, average='macro',  zero_division=0)
test_f1_weighted = f1_score(test_labels, test_preds, average='weighted', zero_division=0)

print(f"\n{'='*50}")
print(f"Test Accuracy        : {test_acc*100:.2f}%")
print(f"Test F1 (macro)      : {test_f1_macro:.4f}")
print(f"Test F1 (weighted)   : {test_f1_weighted:.4f}")
print(f"{'='*50}")

# Per-class report (top 20 classes by support)
report = classification_report(test_labels, test_preds,
                                target_names=class_names, zero_division=0)
print("\nClassification Report (all classes):")
print(report)

# Confusion matrix (show top-30 most frequent classes for readability)
from collections import Counter
top_classes = [c for c, _ in Counter(test_labels.tolist()).most_common(30)]
top_classes_sorted = sorted(top_classes)
mask = np.isin(test_labels, top_classes_sorted)
cm = confusion_matrix(test_labels[mask], test_preds[mask], labels=top_classes_sorted)
cm_names = [class_names[i] for i in top_classes_sorted]

fig, ax = plt.subplots(figsize=(20, 16))
sns.heatmap(cm, annot=False, fmt='d', cmap='Blues',
            xticklabels=cm_names, yticklabels=cm_names, ax=ax)
ax.set_xlabel('Predicted', fontsize=12)
ax.set_ylabel('True', fontsize=12)
ax.set_title(f'Confusion Matrix (Top-30 classes) - Test Acc: {test_acc*100:.2f}%', fontsize=13)
plt.xticks(rotation=45, ha='right', fontsize=8)
plt.yticks(rotation=0, fontsize=8)
plt.tight_layout()
plt.savefig('/content/confusion_matrix.png', bbox_inches='tight', dpi=150)
plt.show()
print("Confusion matrix saved to /content/confusion_matrix.png")


In [ ]:
# Cell 14: Pie chart of class distribution + metrics bar chart
import matplotlib.pyplot as plt
import numpy as np
from collections import Counter

fig, axes = plt.subplots(1, 2, figsize=(16, 7))
fig.suptitle('ISL Recognition - Dataset & Performance Overview', fontsize=14, fontweight='bold')

# ── Left: Pie chart of test set class distribution (top 15 + other) ────────
label_counts = Counter(test_labels.tolist())
top15 = label_counts.most_common(15)
top15_names  = [class_names[c] for c, _ in top15]
top15_counts = [cnt for _, cnt in top15]
other_count  = sum(cnt for c, cnt in label_counts.items()
                   if c not in [x[0] for x in top15])
if other_count > 0:
    top15_names.append('Other')
    top15_counts.append(other_count)

wedge_props = {'edgecolor': 'white', 'linewidth': 1.5}
axes[0].pie(top15_counts, labels=top15_names, autopct='%1.1f%%',
            startangle=140, wedgeprops=wedge_props,
            textprops={'fontsize': 8})
axes[0].set_title('Test Set Class Distribution (Top 15)', fontsize=11)

# ── Right: Metrics bar chart ────────────────────────────────────────────────
metrics = {
    'Accuracy': test_acc * 100,
    'F1 Macro': test_f1_macro * 100,
    'F1 Weighted': test_f1_weighted * 100,
}
colors = ['#2196F3', '#4CAF50', '#FF9800']
bars = axes[1].bar(metrics.keys(), metrics.values(), color=colors, width=0.5, edgecolor='white')
axes[1].set_ylim(0, 110)
axes[1].set_ylabel('Score (%)', fontsize=11)
axes[1].set_title('Model Performance Metrics', fontsize=11)
axes[1].grid(axis='y', alpha=0.3)
for bar, val in zip(bars, metrics.values()):
    axes[1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1.5,
                 f'{val:.1f}%', ha='center', va='bottom', fontweight='bold', fontsize=11)

plt.tight_layout()
plt.savefig('/content/metrics_overview.png', bbox_inches='tight', dpi=150)
plt.show()
print("Metrics overview saved to /content/metrics_overview.png")


In [ ]:
# Cell 15: Per-class F1 bar chart
import matplotlib.pyplot as plt
import numpy as np
from sklearn.metrics import f1_score

# Compute per-class F1
per_class_f1 = f1_score(test_labels, test_preds, average=None,
                         labels=list(range(NUM_CLASSES)), zero_division=0)

# Sort by F1 score descending
sorted_idx = np.argsort(per_class_f1)[::-1]
sorted_f1  = per_class_f1[sorted_idx]
sorted_names = [class_names[i] for i in sorted_idx]

# Plot top-40 and bottom-20 classes
fig, axes = plt.subplots(2, 1, figsize=(20, 14))
fig.suptitle('Per-Class F1 Score - Multimodal BiLSTM', fontsize=14, fontweight='bold')

# Top 40
n_top = min(40, len(sorted_names))
colors_top = plt.cm.RdYlGn(sorted_f1[:n_top])
axes[0].bar(range(n_top), sorted_f1[:n_top] * 100, color=colors_top, edgecolor='white')
axes[0].set_xticks(range(n_top))
axes[0].set_xticklabels(sorted_names[:n_top], rotation=45, ha='right', fontsize=8)
axes[0].set_ylabel('F1 Score (%)')
axes[0].set_title(f'Top {n_top} Classes by F1 Score')
axes[0].set_ylim(0, 110)
axes[0].axhline(y=test_f1_macro*100, color='navy', linestyle='--', alpha=0.7,
                label=f'Macro F1 = {test_f1_macro*100:.1f}%')
axes[0].legend()
axes[0].grid(axis='y', alpha=0.3)

# Bottom 20
n_bot = min(20, len(sorted_names))
bottom_idx   = sorted_idx[-n_bot:][::-1]
bottom_f1    = per_class_f1[bottom_idx]
bottom_names = [class_names[i] for i in bottom_idx]
colors_bot   = plt.cm.RdYlGn(bottom_f1)
axes[1].bar(range(n_bot), bottom_f1 * 100, color=colors_bot, edgecolor='white')
axes[1].set_xticks(range(n_bot))
axes[1].set_xticklabels(bottom_names, rotation=45, ha='right', fontsize=8)
axes[1].set_ylabel('F1 Score (%)')
axes[1].set_title(f'Bottom {n_bot} Classes by F1 Score (hardest classes)')
axes[1].set_ylim(0, 110)
axes[1].axhline(y=test_f1_macro*100, color='navy', linestyle='--', alpha=0.7,
                label=f'Macro F1 = {test_f1_macro*100:.1f}%')
axes[1].legend()
axes[1].grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig('/content/per_class_f1.png', bbox_inches='tight', dpi=150)
plt.show()
print("Per-class F1 chart saved to /content/per_class_f1.png")

# Summary stats
print(f"\nPer-class F1 statistics:")
print(f"  Mean  : {per_class_f1.mean()*100:.2f}%")
print(f"  Median: {np.median(per_class_f1)*100:.2f}%")
print(f"  Min   : {per_class_f1.min()*100:.2f}% ({class_names[per_class_f1.argmin()]})")
print(f"  Max   : {per_class_f1.max()*100:.2f}% ({class_names[per_class_f1.argmax()]})")
print(f"  Classes with F1=0: {(per_class_f1==0).sum()}")


In [ ]:
# Cell 16: Download model and plots to local machine
import shutil, os
from google.colab import files

# Copy checkpoint to /content for easy download
LOCAL_MODEL_PATH = '/content/bilstm_isl_final.pth'
if os.path.exists(CHECKPOINT_PATH):
    shutil.copy(CHECKPOINT_PATH, LOCAL_MODEL_PATH)
    print(f"Model copied to {LOCAL_MODEL_PATH}")

# Files to download
download_files = [
    LOCAL_MODEL_PATH,
    '/content/training_curves.png',
    '/content/confusion_matrix.png',
    '/content/metrics_overview.png',
    '/content/per_class_f1.png',
]

for fpath in download_files:
    if os.path.exists(fpath):
        print(f"Downloading: {fpath}")
        files.download(fpath)
    else:
        print(f"File not found (skipping): {fpath}")

print("\nAll downloads initiated.")
print(f"\nFinal model summary:")
print(f"  Architecture : Multimodal BiLSTM (Pose + MobileNetV2)")
print(f"  Input dim    : 1414 (134 keypoints + 1280 CNN)")
print(f"  Sequence len : {MAX_SEQ_LEN} frames")
print(f"  Classes      : {NUM_CLASSES}")
print(f"  Test Accuracy: {test_acc*100:.2f}%")
print(f"  Test F1 Macro: {test_f1_macro*100:.2f}%")
